In [2]:
# ============================================
# CONVERT CHEST X-RAY IMAGES TO CSV
# ============================================

import numpy as np
import pandas as pd
import os
import cv2
from glob import glob
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from skimage.feature import hog, local_binary_pattern
from skimage.transform import resize
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

np.random.seed(42)

# ============================================
# 1. SETUP PATHS
# ============================================

BASE_PATH = '/kaggle/input/datasets/mtishakil'

# Find the actual dataset folder
print("🔍 Finding dataset...")
current_path = BASE_PATH

for item in os.listdir(current_path):
    item_path = os.path.join(current_path, item)
    if os.path.isdir(item_path):
        sub_items = os.listdir(item_path)
        sub_items_lower = [s.lower() for s in sub_items]
        if any('train' in s for s in sub_items_lower):
            BASE_PATH = item_path
            break
        for sub_item in sub_items:
            sub_path = os.path.join(item_path, sub_item)
            if os.path.isdir(sub_path):
                deep_items = os.listdir(sub_path)
                deep_items_lower = [d.lower() for d in deep_items]
                if any('train' in d for d in deep_items_lower):
                    BASE_PATH = sub_path
                    break

print(f"✅ Dataset found at: {BASE_PATH}")

# Find train folder
train_dir = None
for item in os.listdir(BASE_PATH):
    item_path = os.path.join(BASE_PATH, item)
    if os.path.isdir(item_path) and 'train' in item.lower():
        train_dir = item_path
        break

if train_dir is None:
    train_dir = BASE_PATH

print(f"✅ Train folder: {train_dir}")

# ============================================
# 2. LOAD AND PREPROCESS IMAGES
# ============================================

def load_image(image_path, target_size=(128, 128)):
    """Load and preprocess a single image"""
    try:
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None
        img = resize(img, target_size, anti_aliasing=True)
        img = img.astype(np.float32) / 255.0
        return img
    except:
        return None

print("\n📂 Loading images...")

images = []
labels = []
image_names = []

for class_folder in os.listdir(train_dir):
    class_path = os.path.join(train_dir, class_folder)
    if os.path.isdir(class_path):
        folder_lower = class_folder.lower()
        
        if 'normal' in folder_lower:
            label = 0
            class_name = "Normal"
        elif any(term in folder_lower for term in ['pneumonia', 'bacteria', 'virus']):
            label = 1
            class_name = "Pneumonia"
        else:
            continue
        
        image_paths = []
        for file in os.listdir(class_path):
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.jfif', '.tiff', '.bmp')):
                image_paths.append(os.path.join(class_path, file))
        
        print(f"   Loading {class_name}: {len(image_paths)} images")
        
        for path in tqdm(image_paths, desc=f"   {class_name}"):
            img = load_image(path)
            if img is not None:
                images.append(img)
                labels.append(label)
                image_names.append(os.path.basename(path))

print(f"\n✅ Loaded {len(images)} images successfully")

# ============================================
# 3. EXTRACT FEATURES
# ============================================

print("\n🔍 Extracting features...")

# HOG Features
print("   • HOG features...")
hog_features = []
for img in tqdm(images, desc="     HOG"):
    fd, _ = hog(img, orientations=9, pixels_per_cell=(8, 8),
               cells_per_block=(2, 2), visualize=True, channel_axis=None)
    hog_features.append(fd)
hog_features = np.array(hog_features)
print(f"     ✅ HOG: {hog_features.shape[1]} features")

# LBP Features (Fixed size)
print("   • LBP features...")
lbp_features = []
for img in tqdm(images, desc="     LBP"):
    img_uint8 = (img * 255).astype(np.uint8)
    lbp = local_binary_pattern(img_uint8, 24, 3, method='uniform')
    hist, _ = np.histogram(lbp, bins=26, range=(0, 26), density=True)
    lbp_features.append(hist)
lbp_features = np.array(lbp_features)
print(f"     ✅ LBP: {lbp_features.shape[1]} features")

# Statistical Features
print("   • Statistical features...")
stat_features = []
for img in tqdm(images, desc="     Statistical"):
    features = [
        np.mean(img), np.std(img), np.median(img),
        np.min(img), np.max(img),
        np.percentile(img, 25), np.percentile(img, 75),
        float(pd.Series(img.flatten()).skew()),
        float(pd.Series(img.flatten()).kurtosis())
    ]
    hist, _ = np.histogram(img.flatten(), bins=50, density=True)
    features.append(np.sum(hist ** 2))  # Energy
    features.append(-np.sum(hist * np.log2(hist + 1e-10)))  # Entropy
    stat_features.append(features)
stat_features = np.array(stat_features)
print(f"     ✅ Statistical: {stat_features.shape[1]} features")

# Combine all features
print("\n   Combining features...")
all_features = np.hstack([hog_features, lbp_features, stat_features])
print(f"   ✅ Total features: {all_features.shape[1]}")

# ============================================
# 4. SCALE AND REDUCE FEATURES
# ============================================

print("\n📏 Scaling features...")
scaler = StandardScaler()
scaled_features = scaler.fit_transform(all_features)
print("   ✅ Features scaled")

print("\n📉 Applying PCA...")
pca = PCA(n_components=200)
reduced_features = pca.fit_transform(scaled_features)
print(f"   ✅ Features reduced from {all_features.shape[1]} to {reduced_features.shape[1]}")
print(f"   ✅ Explained variance: {pca.explained_variance_ratio_.sum():.3f} ({pca.explained_variance_ratio_.sum()*100:.1f}%)")

# ============================================
# 5. CREATE DATAFRAME
# ============================================

print("\n📊 Creating CSV file...")

# Create column names
feature_columns = [f'Feature_{i+1}' for i in range(reduced_features.shape[1])]

# Create DataFrame
df = pd.DataFrame(reduced_features, columns=feature_columns)

# Add metadata columns
df.insert(0, 'Image_Name', image_names)
df.insert(1, 'Label', labels)
df.insert(2, 'Class', ['Normal' if l == 0 else 'Pneumonia' for l in labels])

print(f"✅ DataFrame created with {df.shape[0]} rows and {df.shape[1]} columns")

# ============================================
# 6. DISPLAY FIRST 20 ROWS
# ============================================

print("\n" + "="*80)
print("📋 FIRST 20 ROWS OF THE DATASET")
print("="*80)
print(f"\nShowing first 20 rows (out of {len(df)} total rows)")
print(f"Columns: {df.shape[1]} (Image_Name, Label, Class, Feature_1 to Feature_{df.shape[1]-3})")
print("\n")

# Display first 20 rows with selected columns
display_df = df.head(20)
print(display_df.to_string())

# Also show summary of all feature columns
print("\n" + "="*80)
print("📊 DATASET SUMMARY")
print("="*80)
print(f"\nTotal Images: {len(df)}")
print(f"Normal Images: {len(df[df['Label'] == 0])}")
print(f"Pneumonia Images: {len(df[df['Label'] == 1])}")
print(f"Total Features: {df.shape[1] - 3} (excluding Image_Name, Label, Class)")
print(f"\nFeature Statistics:")
print(df[feature_columns].describe())

# ============================================
# 7. SAVE TO CSV
# ============================================

csv_filename = 'chest_xray_features.csv'
df.to_csv(csv_filename, index=False)
print(f"\n✅ CSV file saved as: {csv_filename}")
print(f"   File size: {os.path.getsize(csv_filename) / (1024*1024):.2f} MB")

# ============================================
# 8. CREATE DOWNLOAD LINK
# ============================================

from IPython.display import FileLink, display

print("\n📥 DOWNLOAD LINK:")
print("-"*40)
display(FileLink(csv_filename))
print("\nClick the link above to download the CSV file!")

# Also show how to read it back
print("\n" + "="*80)
print("💡 HOW TO USE THE CSV FILE")
print("="*80)
print("""
# To read the CSV file:
df = pd.read_csv('chest_xray_features.csv')

# Separate features and labels:
X = df.drop(['Image_Name', 'Label', 'Class'], axis=1)
y = df['Label']

# Train your model:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X, y)
""")

print("\n✅ CONVERSION COMPLETE!")

🔍 Finding dataset...
✅ Dataset found at: /kaggle/input/datasets/mtishakil/pneumonia/chest_xray_tvt
✅ Train folder: /kaggle/input/datasets/mtishakil/pneumonia/chest_xray_tvt/train

📂 Loading images...
   Loading Pneumonia: 2993 images


   Pneumonia: 100%|██████████| 2993/2993 [02:18<00:00, 21.63it/s]


   Loading Normal: 1109 images


   Normal: 100%|██████████| 1109/1109 [02:59<00:00,  6.19it/s]



✅ Loaded 4102 images successfully

🔍 Extracting features...
   • HOG features...


...

     ✅ LBP: 26 features
   • Statistical features...


     Statistical: 100%|██████████| 4102/4102 [00:06<00:00, 594.32it/s]


     ✅ Statistical: 11 features

   Combining features...
   ✅ Total features: 8137

📏 Scaling features...
   ✅ Features scaled

📉 Applying PCA...
   ✅ Features reduced from 8137 to 200
   ✅ Explained variance: 0.548 (54.8%)

📊 Creating CSV file...
✅ DataFrame created with 4102 rows and 203 columns

📋 FIRST 20 ROWS OF THE DATASET

Showing first 20 rows (out of 4102 total rows)
Columns: 203 (Image_Name, Label, Class, Feature_1 to Feature_200)


                    Image_Name  Label      Class  Feature_1  Feature_2  Feature_3  Feature_4  Feature_5  Feature_6  Feature_7  Feature_8  Feature_9  Feature_10  Feature_11  Feature_12  Feature_13  Feature_14  Feature_15  Feature_16  Feature_17  Feature_18  Feature_19  Feature_20  Feature_21  Feature_22  Feature_23  Feature_24  Feature_25  Feature_26  Feature_27  Feature_28  Feature_29  Feature_30  Feature_31  Feature_32  Feature_33  Feature_34  Feature_35  Feature_36  Feature_37  Feature_38  Feature_39  Feature_40  Feature_41  Feature_42  Feature

/kaggle/working/chest_xray_features.csv


Click the link above to download the CSV file!

💡 HOW TO USE THE CSV FILE

# To read the CSV file:
df = pd.read_csv('chest_xray_features.csv')

# Separate features and labels:
X = df.drop(['Image_Name', 'Label', 'Class'], axis=1)
y = df['Label']

# Train your model:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X, y)


✅ CONVERSION COMPLETE!



**Raw Image (.jpeg)    ↓ load_image()128×128 Array (16384 pixels)    ↓ extract_features()8137 Features    ↓ StandardScaler8137 Scaled Features    ↓ PCA200 Features    ↓ DataFrameCSV File (.csv)**
